In [1]:
import os
from dataclasses import dataclass

from common import inspect_response , configure_notebook


@dataclass(frozen=True)
class OpenRouterConfig:
    model: str
    base_url: str
    api_key: str
    env_file: str

    def summary(self) -> str:
        masked_key = f"{self.api_key[:6]}...{self.api_key[-4:]}"
        return (
            f"env file : {self.env_file}\n"
            f"base url : {self.base_url}\n"
            f"model    : {self.model}\n"
            f"api key  : {masked_key}"
        )


env_path = configure_notebook()

config = OpenRouterConfig(
    model="moonshotai/kimi-k2.6",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY", ""),
    env_file=env_path.name,
)

assert config.api_key, "Add OPENROUTER_API_KEY to .env first."
assert not config.model.startswith("replace-"), "Set the model in this cell."
print(config.summary())

env file : .env
base url : https://openrouter.ai/api/v1
model    : moonshotai/kimi-k2.6
api key  : sk-or-...34fb


## A. Basic API Call

In [2]:
# Previously used: Chat Completions API - `client.chat.completions`
# Kept here only for comparison with the Responses API.
# Official docs: https://developers.openai.com/api/reference/python/resources/chat/subresources/completions/methods/create

# Recommended for new projects: Responses API
# OpenAI recommends using the Responses API for new projects.
# Official docs and migration guide:
# https://developers.openai.com/api/docs/guides/migrate-to-responses

from openai import OpenAI

client = OpenAI(
    api_key=config.api_key,
    base_url=config.base_url,
)

response = client.responses.create(
    model=config.model,
    instructions="You are a helpful assistant.",
    input="Hello who are you",
)

inspect_result = inspect_response(response)
# print(inspect_result) - comment off to see the json


════════════════════════════════════════════════════════════════════════
                       RESPONSES API · INSPECTION                       
════════════════════════════════════════════════════════════════════════

STATE
────────────────────────────────────────────────────────────────────────
  status                    completed
  model                     moonshotai/kimi-k2.6
  server duration           2.000 s

INPUT
────────────────────────────────────────────────────────────────────────
  request input             not supplied; pass request_input=the_input_used_for_create

OUTPUT · ASSISTANT
────────────────────────────────────────────────────────────────────────
Hello! I'm an AI assistant created by Moonshot AI. I'm here to help you with a wide range of tasks, from answering questions and writing assistance to coding, analysis, and more. How can I help you today?

INTERMEDIATE RESULTS · REASONING
────────────────────────────────────────────────────────────────────────
  ret

## B. Structured Response

In [4]:
# ============================================================
# Concept: Structured Response with Pydantic
# ============================================================

# Official docs:
# https://developers.openai.com/api/docs/guides/structured-outputs

# Problem:
# Asking in the prompt, "return JSON like this", is not a strong contract.
# The model may follow it, but prompt-only JSON is not 100% guaranteed.

# Solution:
# Use Pydantic class to create structured output schema
# Pass that class to `text_format`.
# The SDK uses it as the structured output schema.

# Result:
# `response.output_parsed` gives a normal validated Python object.

from openai import OpenAI
from pydantic import BaseModel

client = OpenAI(
    api_key=config.api_key,
    base_url=config.base_url,
)


class TopicExplanation(BaseModel):
    topic: str
    key_idea: str
    why_it_matters: str


try:
    response = client.responses.parse(
        model=config.model,
        instructions="Explain the topic simply for a Software Engineer.",
        input="Explain LLM.",
        text_format=TopicExplanation,
    )
except Exception as exc:
    reason = f"API call failed ({type(exc).__name__}): {exc}"
    print(reason)
    

result = response.output_parsed
inspect_result = inspect_response(response)

print("\n\n")
print("Actual Response:")
print(f"Topic: {result.topic}")
print(f"Key Idea: {result.key_idea}")
print(f"Why It Matters: {result.why_it_matters}")


════════════════════════════════════════════════════════════════════════
                       RESPONSES API · INSPECTION                       
════════════════════════════════════════════════════════════════════════

STATE
────────────────────────────────────────────────────────────────────────
  status                    completed
  model                     moonshotai/kimi-k2.6
  server duration           29.000 s

INPUT
────────────────────────────────────────────────────────────────────────
  request input             not supplied; pass request_input=the_input_used_for_create

OUTPUT · ASSISTANT
────────────────────────────────────────────────────────────────────────
{ "key_idea": "An LLM is a stateless, statistical next-token predictor built on transformer architecture—essentially a massive function `f(tokens, weights) -> probability_distribution` that generates text by repeatedly sampling the most likely next token." , "topic": "LLM", "why_it_matters": "Understanding LLMs as 

## C. Streaming

In [5]:
# Official Docs : 
# About Streaming API
# https://developers.openai.com/api/docs/guides/streaming-responses?api-mode=responses
# All the Streaming API's events possible 
# Original Docs : https://developers.openai.com/api/reference/resources/responses/streaming-events

from openai import OpenAI
import json
from common import inspect_response_stream

client = OpenAI(
    api_key=config.api_key,
    base_url=config.base_url,
)

request_input = "Explain transformers in 5 short points."

stream = client.responses.create(
    model=config.model,
    input=request_input,
    stream=True,
)


# The parser consumes the stream and prints the generated text live.
stream_result = inspect_response_stream(
    stream,
    request_input=request_input,
)

# Print every captured event so you can inspect its fields.
print("\n\n\n")
print("\n\nRAW STREAM EVENTS")

# for event in stream_result["stream"]["events"]:
#     print(json.dumps(event, indent=2))



INPUT · STREAM REQUEST
────────────────────────────────────────────────────────────────────────
  request input             Explain transformers in 5 short points.

LIVE REASONING · output[0] part[0]
────────────────────────────────────────────────────────────────────────
 The user wants an explanation of transformers in 5 short points. I need to be concise, clear, and accurate. Key aspects to cover:

1. Core mechanism (attention/self-attention)
2. Architecture (encoder-decoder, parallel processing)
3. Key advantage over RNNs (no sequential processing, long-range dependencies)
4. Components (multi-head attention, positional encoding, feed-forward)
5. Impact/applications (NLP, GPT, BERT, etc.)

Let me draft 5 short points:

1. **Attention mechanism**: They rely on "self-attention" to weigh the importance of different words/tokens in a sequence simultaneously, capturing relationships regardless of distance.

2. **Parallel processing**: Unlike RNNs, transformers process entire sequences 